# SIH26012 — AI-Based Automated Urban Parcel Mapping
## Master Colab Pro Training: Full Integrated Architecture
### Progressive Multiscale Generator (PMG) + Domain Generalization (DG) + Connectivity Dual-Head + Residual U-Net

This notebook provides the complete, production training and evaluation pipeline for **Smart India Hackathon (SIH26012)**.

---

### Step 1: Install Dependencies and Verify Hardware Accelerator

In [ ]:
!pip install -q rasterio geopandas shapely pyogrio opencv-python-headless matplotlib pillow pyyaml tqdm networkx tabulate uvicorn fastapi

import os, sys, time, json, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("=== Hardware Accelerator Inspection ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU:      {gpu_name}")
    print(f"Total VRAM:      {vram_gb:.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("Running on CPU / MPS fallback.")

### Step 2: Verify Prepared Dataset Catalogs

In [ ]:
train_csv = "data/processed/metadata/patches_train.csv"
val_csv = "data/processed/metadata/patches_val.csv"
test_csv = "data/processed/metadata/patches_test.csv"

assert os.path.exists(train_csv), "Missing patches_train.csv!"
assert os.path.exists(val_csv), "Missing patches_val.csv!"
assert os.path.exists(test_csv), "Missing patches_test.csv!"

df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)
df_test = pd.read_csv(test_csv)

print(f"[✓] Train Patches:      {len(df_train)} (5 distinct geographic tiles)")
print(f"[✓] Validation Patches: {len(df_val)} (2 distinct geographic tiles)")
print(f"[✓] Test Patches:       {len(df_test)} (2 distinct geographic tiles)")

### Step 3: Launch Full Model Training (PMG + DG + Connectivity)

In [ ]:
# Execute full end-to-end training and evaluation
!python scripts/train_full.py configs/full.yaml

### Step 4: Display Training Curves & Evaluation Metrics

In [ ]:
history_path = "experiments/full/metrics/training_history.json"
test_metrics_path = "experiments/full/metrics/metrics_test.json"

if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    df_hist = pd.DataFrame(history)
    display(df_hist)

if os.path.exists(test_metrics_path):
    with open(test_metrics_path) as f:
        test_m = json.load(f)
    print("\n=== Held-Out Test Set Metrics ===")
    print(f"Test F1 Score:   {test_m['f1']:.4f}")
    print(f"Test IoU:        {test_m['iou']:.4f}")
    print(f"Test Precision:  {test_m['precision']:.4f}")
    print(f"Test Recall:     {test_m['recall']:.4f}")

### Step 5: End-to-End GIS Raster-to-Vector & Topology Audit

In [ ]:
from PIL import Image
from ai.models.full_model import CadastreUNetFull
from gis.pipeline import RasterToVectorGISPipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CadastreUNetFull().to(device)
ckpt_path = "experiments/full/checkpoints/best_model.pth"

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Select demo test patch
sample_row = df_test.iloc[0]
img_path = os.path.join("data/processed", sample_row["image_path"])
img_pil = Image.open(img_path).convert("RGB")
img_tensor = torch.from_numpy(np.array(img_pil).transpose(2, 0, 1) / 255.0).unsqueeze(0).float().to(device)

with torch.no_grad():
    pred_out = model(img_tensor, return_dict=True)
    prob_map = pred_out["refined_prob"][0, 0].cpu().numpy()

pipeline = RasterToVectorGISPipeline(threshold=0.5, simplify_tolerance=0.5)
gis_result = pipeline.run(prob_map, transform=(0.25, 0.0, 100000.0, 0.0, -0.25, 500000.0))

print("=== GIS Vectorization & Topology Report ===")
print(f"Total Extracted Lines:   {gis_result['total_lines']}")
print(f"Total Boundary Length:   {gis_result['total_length_m']} m")
print(f"Topology Status:         {gis_result['topology_report']['status']}")
print(f"Self-Intersections:      {gis_result['topology_report']['self_intersections_count']}")
print(f"Invalid Geometries:      {gis_result['topology_report']['invalid_geometries_count']}")